# Direct YOLO SKU Detector Baseline — Colab GPU Ready

This notebook runs the **real YOLO GPU baseline** for the retail shelf SKU recognition project.

ZIP location:

```text
/content/drive/MyDrive/Transmed/colab_yolo_package.zip
```

The notebook will:

1. Mount Google Drive.
2. Locate and copy the ZIP package.
3. Extract it into `/content/transmid_project`.
4. Fix `data.yaml` path for Colab.
5. Train YOLO on GPU.
6. Evaluate validation and test splits.
7. Generate inference previews.
8. Save the results ZIP back to Google Drive.

## 1. Verify GPU

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not enabled")

CUDA available: True
GPU: Tesla T4


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Configure package path

Set `DRIVE_PACKAGE_FOLDER` to the Google Drive folder that contains your ZIP.  
You told me the ZIP is inside a folder named **Transmed**, so this is already set correctly.

In [3]:
from pathlib import Path
import os, shutil, glob, json, yaml

DRIVE_PACKAGE_FOLDER = Path("/content/drive/MyDrive/Transmed")
PACKAGE_NAME = "colab_yolo_package.zip"

zip_path = DRIVE_PACKAGE_FOLDER / PACKAGE_NAME

print("Expected ZIP path:", zip_path)
print("Exists:", zip_path.exists())

if not zip_path.exists():
    print("\nZIP not found. Searching inside the Transmed folder...")
    found = list(DRIVE_PACKAGE_FOLDER.rglob("*.zip"))
    print("Found ZIP files:")
    for p in found:
        print(" -", p)
    if len(found) == 1:
        zip_path = found[0]
        print("\nUsing:", zip_path)
    else:
        raise FileNotFoundError(
            "Could not uniquely find colab_yolo_package.zip. "
            "Check the ZIP name or update DRIVE_PACKAGE_FOLDER / PACKAGE_NAME."
        )

Expected ZIP path: /content/drive/MyDrive/Transmed/colab_yolo_package.zip
Exists: True


## 4. Copy ZIP to Colab runtime and extract

In [4]:
runtime_zip = Path("/content/colab_yolo_package.zip")
project_dir = Path("/content/transmid_project")

# Clean previous extraction
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

# Copy ZIP from Drive to local Colab runtime for faster I/O
shutil.copy2(zip_path, runtime_zip)
print("Copied ZIP to:", runtime_zip)
print("ZIP size MB:", runtime_zip.stat().st_size / (1024 * 1024))

# Extract
!unzip -q "/content/colab_yolo_package.zip" -d "/content/transmid_project"

print("Extraction complete.")

Copied ZIP to: /content/colab_yolo_package.zip
ZIP size MB: 940.4310836791992
Extraction complete.


## 5. Find project root

The project root should be the folder that directly contains:

```text
data/
ml/
configs/
```

This cell automatically finds it, so it works whether the ZIP extracted directly or inside a nested folder.

In [5]:
from pathlib import Path
import os

def find_project_root(base: Path) -> Path:
    candidates = []
    for p in [base] + [x for x in base.rglob("*") if x.is_dir()]:
        if (p / "data").exists() and (p / "ml").exists() and (p / "configs").exists():
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not find project root containing data/, ml/, configs/.")
    # Prefer shortest path
    candidates = sorted(candidates, key=lambda x: len(str(x)))
    return candidates[0]

PROJECT_ROOT = find_project_root(project_dir)
print("PROJECT_ROOT:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Current working directory:", Path.cwd())

print("\nTop-level files/folders:")
!ls -la

PROJECT_ROOT: /content/transmid_project
Current working directory: /content/transmid_project

Top-level files/folders:
total 24
drwxr-xr-x 5 root root 4096 Jul 10 16:49 .
drwxr-xr-x 1 root root 4096 Jul 10 16:49 ..
drwxr-xr-x 2 root root 4096 Jul 10 16:49 configs
drwxr-xr-x 3 root root 4096 Jul 10 16:49 data
drwxr-xr-x 3 root root 4096 Jul 10 16:49 ml
-rw-r--r-- 1 root root   72 Jul  9 23:15 requirements.txt


## 6. Verify required files

In [6]:
required_paths = [
    "data/processed/yolo_remapped/data.yaml",
    "data/processed/yolo_remapped/images/train",
    "data/processed/yolo_remapped/images/val",
    "data/processed/yolo_remapped/images/test",
    "data/processed/yolo_remapped/labels/train",
    "data/processed/yolo_remapped/labels/val",
    "data/processed/yolo_remapped/labels/test",
    "ml/detection/train_yolo.py",
    "ml/detection/evaluate_yolo.py",
    "ml/detection/infer_detector.py",
    "configs/class_id_mapping.json",
]

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Missing required paths:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Some required files/folders are missing from the package.")

print("All required files/folders found.")

for split in ["train", "val", "test"]:
    n_images = len(list(Path(f"data/processed/yolo_remapped/images/{split}").glob("*")))
    n_labels = len(list(Path(f"data/processed/yolo_remapped/labels/{split}").glob("*.txt")))
    print(f"{split}: {n_images} images, {n_labels} label files")

All required files/folders found.
train: 703 images, 703 label files
val: 147 images, 147 label files
test: 150 images, 150 label files


## 7. Install dependencies

In [7]:
!pip install -q ultralytics opencv-python pyyaml pandas numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:0000:01


## 8. Fix `data.yaml` path for Colab

In [8]:
from pathlib import Path
import yaml

data_yaml = Path("data/processed/yolo_remapped/data.yaml")

with open(data_yaml, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(Path("data/processed/yolo_remapped").resolve())

# Ensure nc exists and is correct
if "names" in cfg:
    cfg["nc"] = len(cfg["names"])

with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Updated data.yaml:")
print(yaml.safe_dump(cfg, sort_keys=False))

Updated data.yaml:
path: /content/transmid_project/data/processed/yolo_remapped
train: images/train
val: images/val
test: images/test
nc: 67
names:
  0: old_class_0
  1: old_class_1
  2: old_class_2
  3: old_class_3
  4: old_class_4
  5: old_class_5
  6: old_class_6
  7: old_class_7
  8: old_class_8
  9: old_class_9
  10: old_class_10
  11: old_class_11
  12: old_class_12
  13: old_class_13
  14: old_class_14
  15: old_class_15
  16: old_class_16
  17: old_class_17
  18: old_class_18
  19: old_class_19
  20: old_class_20
  21: old_class_24
  22: old_class_25
  23: old_class_26
  24: old_class_27
  25: old_class_28
  26: old_class_29
  27: old_class_30
  28: old_class_31
  29: old_class_32
  30: old_class_33
  31: old_class_34
  32: old_class_35
  33: old_class_36
  34: old_class_38
  35: old_class_41
  36: old_class_42
  37: old_class_43
  38: old_class_44
  39: old_class_45
  40: old_class_46
  41: old_class_47
  42: old_class_48
  43: old_class_49
  44: old_class_54
  45: old_class_5

## 9. Run a quick dataset sanity check

In [9]:
import yaml
from pathlib import Path

with open("data/processed/yolo_remapped/data.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

assert cfg["nc"] == 67, f"Expected nc=67, got {cfg['nc']}"
assert Path(cfg["path"]).exists(), f"Dataset path does not exist: {cfg['path']}"

bad_labels = []
for split in ["train", "val", "test"]:
    for label_file in Path(f"data/processed/yolo_remapped/labels/{split}").glob("*.txt"):
        for line_no, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), start=1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                bad_labels.append((str(label_file), line_no, "wrong token count", line))
                continue
            cls = int(float(parts[0]))
            if cls < 0 or cls > 66:
                bad_labels.append((str(label_file), line_no, "class out of range", line))

print("Bad labels:", len(bad_labels))
if bad_labels[:10]:
    for item in bad_labels[:10]:
        print(item)

assert len(bad_labels) == 0, "Found invalid labels. Stop before training."
print("Sanity check passed.")

Bad labels: 0
Sanity check passed.


## 10. Choose training configuration

Default uses `yolov8s.pt`, 50 epochs, batch 16, image size 640 on GPU.

On Tesla T4, this should usually work. If you get CUDA memory error, change:

```python
MODEL = "yolov8n.pt"
BATCH = 8
RUN_NAME = "yolo_baseline_50ep_nano"
```

In [10]:
MODEL = "yolov8s.pt"
EPOCHS = 50
BATCH = 16
IMGSZ = 640
DEVICE = 0
RUN_NAME = "yolo_baseline_50ep"

print({
    "MODEL": MODEL,
    "EPOCHS": EPOCHS,
    "BATCH": BATCH,
    "IMGSZ": IMGSZ,
    "DEVICE": DEVICE,
    "RUN_NAME": RUN_NAME,
})

{'MODEL': 'yolov8s.pt', 'EPOCHS': 50, 'BATCH': 16, 'IMGSZ': 640, 'DEVICE': 0, 'RUN_NAME': 'yolo_baseline_50ep'}


## 11. Train YOLO baseline on GPU

In [11]:
!python ml/detection/train_yolo.py \
    --data data/processed/yolo_remapped/data.yaml \
    --model {MODEL} \
    --epochs {EPOCHS} \
    --device {DEVICE} \
    --batch {BATCH} \
    --imgsz {IMGSZ} \
    --name {RUN_NAME}

Direct YOLO Baseline - Dataset Verification & training launcher
data.yaml:    /content/transmid_project/data/processed/yolo_remapped/data.yaml
Model weight: yolov8s.pt
Epochs count: 50
Image size:   640
Batch size:   16
Device:       0
Output name:  yolo_baseline_50ep

Dataset Verification:
  Dataset path: /content/transmid_project/data/processed/yolo_remapped
  Split 'train': Found 703 images and 703 label files.
  Split 'val': Found 147 images and 147 label files.
  Split 'test': Found 150 images and 150 label files.
PASS: Dataset validation passed successfully! All annotations are formatted correctly.

Generating visual sanity previews in: /content/transmid_project/reports/experiments/yolo_baseline/sanity_previews
  Saved sanity preview: sanity_preview_1_Transmed_Autolabelling642.jpg
  Saved sanity preview: sanity_preview_2_Transmed-TEA-NI029.jpg
  Saved sanity preview: sanity_preview_3_Transmed Others 250.jpg

Importing Ultralytics YOLO to train on device='0'...
Creating new Ultral

## 12. Check training outputs

In [12]:
weights_path = Path(f"runs/detect/{RUN_NAME}/weights/best.pt")
last_weights_path = Path(f"runs/detect/{RUN_NAME}/weights/last.pt")

print("Best weights:", weights_path, "exists:", weights_path.exists())
print("Last weights:", last_weights_path, "exists:", last_weights_path.exists())

if not weights_path.exists():
    raise FileNotFoundError(f"Training did not produce best.pt at {weights_path}")

!ls -la runs/detect/{RUN_NAME}
!ls -la runs/detect/{RUN_NAME}/weights

Best weights: runs/detect/yolo_baseline_50ep/weights/best.pt exists: True
Last weights: runs/detect/yolo_baseline_50ep/weights/last.pt exists: True
total 12912
drwxr-xr-x 3 root root    4096 Jul 10 17:14 .
drwxr-xr-x 3 root root    4096 Jul 10 16:50 ..
-rw-r--r-- 1 root root    1737 Jul 10 16:50 args.yaml
-rw-r--r-- 1 root root  565437 Jul 10 17:14 BoxF1_curve.png
-rw-r--r-- 1 root root  487414 Jul 10 17:14 BoxP_curve.png
-rw-r--r-- 1 root root  171332 Jul 10 17:14 BoxPR_curve.png
-rw-r--r-- 1 root root  456907 Jul 10 17:14 BoxR_curve.png
-rw-r--r-- 1 root root  366458 Jul 10 17:14 confusion_matrix_normalized.png
-rw-r--r-- 1 root root  362980 Jul 10 17:14 confusion_matrix.png
-rw-r--r-- 1 root root  148493 Jul 10 16:50 labels.jpg
-rw-r--r-- 1 root root    6421 Jul 10 17:14 results.csv
-rw-r--r-- 1 root root  234982 Jul 10 17:14 results.png
-rw-r--r-- 1 root root 1029628 Jul 10 16:51 train_batch0.jpg
-rw-r--r-- 1 root root  740032 Jul 10 17:09 train_batch1760.jpg
-rw-r--r-- 1 root root

## 13. Evaluate validation split

In [13]:
!python ml/detection/evaluate_yolo.py \
    --weights runs/detect/{RUN_NAME}/weights/best.pt \
    --data data/processed/yolo_remapped/data.yaml \
    --split val \
    --name yolo_eval_50ep_val

YOLO Baseline Evaluation Runner
Weights:     /content/transmid_project/runs/detect/yolo_baseline_50ep/weights/best.pt
data.yaml:   /content/transmid_project/data/processed/yolo_remapped/data.yaml
Split:       val
Project:     runs/detect
Name:        yolo_eval_50ep_val
Importing YOLO module...
Running validation on split 'val'...
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,151,513 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3607.0±1494.1 MB/s, size: 1125.7 KB)
val: Scanning /content/transmid_project/data/processed/yolo_remapped/labels/val.cache... 147 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 147/147 19.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.6s/it 15.8s.6ss
                   all        147       6258      0.866      0.901      0.933      0.845
           old_class_0     

## 14. Evaluate test split

In [14]:
!python ml/detection/evaluate_yolo.py \
    --weights runs/detect/{RUN_NAME}/weights/best.pt \
    --data data/processed/yolo_remapped/data.yaml \
    --split test \
    --name yolo_eval_50ep_test

YOLO Baseline Evaluation Runner
Weights:     /content/transmid_project/runs/detect/yolo_baseline_50ep/weights/best.pt
data.yaml:   /content/transmid_project/data/processed/yolo_remapped/data.yaml
Split:       test
Project:     runs/detect
Name:        yolo_eval_50ep_test
Importing YOLO module...
Running validation on split 'test'...
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,151,513 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 92.4±62.2 MB/s, size: 642.3 KB)
val: Scanning /content/transmid_project/data/processed/yolo_remapped/labels/test... 150 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 304.9it/s 0.5s0.1s
val: New cache created: /content/transmid_project/data/processed/yolo_remapped/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.5s/it 15.2s.4ss
                   all

## 15. Generate prediction previews

In [15]:
!python ml/detection/infer_detector.py \
    --weights runs/detect/{RUN_NAME}/weights/best.pt \
    --source data/processed/yolo_remapped/images/test \
    --output-dir reports/experiments/yolo_baseline/inference_previews_50ep \
    --num-samples 20 \
    --conf 0.25 \
    --imgsz 640

YOLO Inference Previews Runner
Weights:     /content/transmid_project/runs/detect/yolo_baseline_50ep/weights/best.pt
Source:      /content/transmid_project/data/processed/yolo_remapped/images/test
Output Dir:  /content/transmid_project/reports/experiments/yolo_baseline/inference_previews_50ep
Samples count:20
Confidence:  0.25

Running predictions on 20 sampled images...
  Predicting [1/20]: Transmed-TEA-NI028.jpg

image 1/1 /content/transmid_project/data/processed/yolo_remapped/images/test/Transmed-TEA-NI028.jpg: 640x480 3 old_class_0s, 3 old_class_2s, 4 old_class_4s, 2 old_class_5s, 1 old_class_10, 7 old_class_13s, 3 old_class_14s, 4 old_class_15s, 3 old_class_16s, 3 old_class_17s, 3 old_class_18s, 3 old_class_20s, 3 old_class_24s, 3 old_class_26s, 5 old_class_32s, 2 old_class_34s, 6 old_class_35s, 3 old_class_36s, 12 old_class_38s, 3 old_class_41s, 4 old_class_42s, 1 old_class_43, 5 old_class_44s, 3 old_class_47s, 6 old_class_59s, 1 old_class_73, 325.9ms
Speed: 5.3ms preprocess, 325

## 16. Inspect result files

In [16]:
print("Training run files:")
!find runs/detect/{RUN_NAME} -maxdepth 2 -type f | head -50

print("\nYOLO baseline report folder:")
!find reports/experiments/yolo_baseline -maxdepth 3 -type f | head -80

Training run files:
runs/detect/yolo_baseline_50ep/val_batch1_labels.jpg
runs/detect/yolo_baseline_50ep/val_batch2_labels.jpg
runs/detect/yolo_baseline_50ep/results.png
runs/detect/yolo_baseline_50ep/val_batch1_pred.jpg
runs/detect/yolo_baseline_50ep/confusion_matrix.png
runs/detect/yolo_baseline_50ep/train_batch0.jpg
runs/detect/yolo_baseline_50ep/val_batch0_labels.jpg
runs/detect/yolo_baseline_50ep/args.yaml
runs/detect/yolo_baseline_50ep/confusion_matrix_normalized.png
runs/detect/yolo_baseline_50ep/BoxR_curve.png
runs/detect/yolo_baseline_50ep/labels.jpg
runs/detect/yolo_baseline_50ep/BoxF1_curve.png
runs/detect/yolo_baseline_50ep/BoxPR_curve.png
runs/detect/yolo_baseline_50ep/train_batch2.jpg
runs/detect/yolo_baseline_50ep/weights/last.pt
runs/detect/yolo_baseline_50ep/weights/best.pt
runs/detect/yolo_baseline_50ep/val_batch2_pred.jpg
runs/detect/yolo_baseline_50ep/train_batch1762.jpg
runs/detect/yolo_baseline_50ep/train_batch1761.jpg
runs/detect/yolo_baseline_50ep/train_batch1.jp

## 17. Zip results and copy back to Google Drive

In [17]:
RESULTS_ZIP = f"{RUN_NAME}_results.zip"
DRIVE_RESULTS_PATH = DRIVE_PACKAGE_FOLDER / RESULTS_ZIP

# Include training run, eval runs, and reports if present.
!zip -r "{RESULTS_ZIP}" \
    "runs/detect/{RUN_NAME}" \
    "runs/detect/yolo_eval_50ep_val" \
    "runs/detect/yolo_eval_50ep_test" \
    "reports/experiments/yolo_baseline" \
    -q

shutil.copy2(RESULTS_ZIP, DRIVE_RESULTS_PATH)

print("Saved results ZIP to:", DRIVE_RESULTS_PATH)
print("Size MB:", DRIVE_RESULTS_PATH.stat().st_size / (1024 * 1024))

Saved results ZIP to: /content/drive/MyDrive/Transmed/yolo_baseline_50ep_results.zip
Size MB: 92.53922271728516
